In [26]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/llm3_mt5_bestmodel/pytorch/default/1/best_model (7).pth
/kaggle/input/test-data/UNLABELLED_TEST.csv
/kaggle/input/logicloom/LABELLED_DEV.csv
/kaggle/input/logicloom/LABELLED_TRAIN.csv


# Imports and Installations

In [27]:
!pip install transformers accelerate evaluate rouge_score

/usr/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [28]:
pip install nltk beautifulsoup4 langdetect

Note: you may need to restart the kernel to use updated packages.


In [29]:
from datasets import load_dataset
train = load_dataset("csv",data_files="/kaggle/input/logicloom/LABELLED_TRAIN.csv")

In [30]:
print(type(train))

<class 'datasets.dataset_dict.DatasetDict'>


In [31]:
eval = load_dataset("csv",data_files="/kaggle/input/logicloom/LABELLED_DEV.csv")

In [32]:
train_dataset = train.filter(lambda example: example["News Article"] is not None and example["Caption"] is not None)

In [33]:
eval_dataset = eval.filter(lambda example: example["News Article"] is not None and example["Caption"] is not None)

# Data Pre-processing

In [34]:
import re
from bs4 import BeautifulSoup
def clean_text(text):
    text = re.sub(r"<.*?>", "", text)    # Remove HTML tags
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE) #remove URL's
    text = re.sub(r"[^a-zA-Z0-9\s.,!?]", "", text)  # Remove special characters
    return text

In [35]:
from langdetect import detect
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stopword_sets = {lang: set(stopwords.words(lang)) for lang in stopwords.fileids()}

def preprocess_text(text):
    text = clean_text(text)
    
    # Detect language
    lang = detect(text)
    
    # Remove stopwords (if language is known and supported)
    if lang in stopword_sets:
        tokens = word_tokenize(text)
        text = " ".join([word for word in tokens if word.lower() not in stopword_sets[lang]])
    
    # Normalize whitespace and lowercase
    text = " ".join(text.split())
    text = text.lower()

    return text


In [36]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def preprocess_function(examples):
    inputs = [preprocess_text(text) for text in examples["News Article"]]
    targets = [preprocess_text(text) for text in examples["Caption"]]

    model_inputs = tokenizer(inputs, max_length=256, truncation=True,padding="max_length",
                             return_tensors="pt")
    labels = tokenizer(targets, max_length=64, truncation=True,padding="max_length",
                      return_tensors="pt")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset_train = train_dataset.map(preprocess_function, batched=True)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [37]:
tokenized_dataset_eval = eval_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [38]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


# Training Model - mt5-small

In [39]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer,DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    logging_dir='./logs',
)

In [40]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_eval,
    tokenizer=tokenizer,
)
# trainer.train()

In [41]:
from tqdm import tqdm
import torch
from torch.optim import Adam

# Initialize the optimizer
optimizer = Adam(model.parameters(), lr=5e-4)  # Adjust learning rate as needed

# Use the dataset already tokenized and passed to the trainer
# train_dataset = tokenized_dataset_train

# Initialize the data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

# Create DataLoader
train_dataloader = torch.utils.data.DataLoader(
    tokenized_dataset_train["train"],
    batch_size=8,  # Adjust batch size as needed
    collate_fn=lambda batch: {
        key: torch.tensor([item[key] for item in batch], dtype=torch.long) 
        if isinstance(batch[0].get(key), list) else batch[0][key] 
        for key in batch[0]
    }
)
val_dataloader = torch.utils.data.DataLoader(
    tokenized_dataset_eval["train"],
    batch_size=8,  # Adjust batch size as needed
    collate_fn=lambda batch: {
        key: torch.tensor([item[key] for item in batch], dtype=torch.long) 
        if isinstance(batch[0].get(key), list) else batch[0][key] 
        for key in batch[0]
    }
)

model = trainer.model
model.train()


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [42]:
print(type(tokenized_dataset_train))
tokenized_dataset_train

<class 'datasets.dataset_dict.DatasetDict'>


DatasetDict({
    train: Dataset({
        features: ['ID', 'News Article', 'Caption', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})

In [43]:
print(type(tokenized_dataset_eval))
tokenized_dataset_eval

<class 'datasets.dataset_dict.DatasetDict'>


DatasetDict({
    train: Dataset({
        features: ['ID', 'News Article', 'Caption', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})

In [44]:
# import torch

# torch.cuda.empty_cache()


In [45]:
# Initialize the best validation loss to a large number
best_val_loss = float('inf')

# Custom training loop
for epoch in range(int(training_args.num_train_epochs)):
    print(f"Epoch {epoch + 1}/{int(training_args.num_train_epochs)}")
    epoch_loss = 0  # To track the epoch loss
    
    for batch in tqdm(train_dataloader):
        # Move data to the appropriate device (GPU/CPU)
        batch = {k: v.to(training_args.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
      
        # Remove any keys that are not needed by the model
        batch = {k: v for k, v in batch.items() if k in ["input_ids", "attention_mask", "labels"]}
   
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        epoch_loss += loss.item()

        # Backward pass
        loss.backward()

        # Optimizer step
        optimizer.step()
        optimizer.zero_grad()

    # Print training loss after each epoch
    print(f"Training loss after epoch {epoch + 1}: {epoch_loss / len(train_dataloader)}")
    
    # Now calculate validation loss after each epoch
    model.eval()  # Set model to evaluation mode
    val_loss = 0
    with torch.no_grad():  # Disable gradients for validation to save memory
        for batch in tqdm(val_dataloader):
            batch = {k: v.to(training_args.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}  

            batch = {k: v for k, v in batch.items() if k in ["input_ids", "attention_mask", "labels"]}

            outputs = model(**batch)
            loss = outputs.loss
            val_loss += loss.item()

    # Print validation loss after each epoch
    print(f"Validation loss after epoch {epoch + 1}: {val_loss / len(val_dataloader)}")
    
    # Check if the current validation loss is better (lower) than the best so far
    if val_loss / len(val_dataloader) < best_val_loss:
        best_val_loss = val_loss / len(val_dataloader)
        print(f"Validation loss decreased to {best_val_loss}. Saving model...")

        # Save the model's state_dict and tokenizer
        torch.save(model.state_dict(), "best_model.pth")  # Save the model state_dict
        tokenizer.save_pretrained("best_model")  # Save the tokenizer

        # Optionally, you can also save the optimizer if you want to resume training later
        torch.save(optimizer.state_dict(), "best_model_optimizer.pt")
        print("Best model saved.")

Epoch 1/5


100%|██████████| 375/375 [02:47<00:00,  2.24it/s]


Training loss after epoch 1: 5.22035808022817


100%|██████████| 125/125 [00:13<00:00,  9.15it/s]


Validation loss after epoch 1: 1.794526463508606
Validation loss decreased to 1.794526463508606. Saving model...
Best model saved.
Epoch 2/5


100%|██████████| 375/375 [02:44<00:00,  2.28it/s]


... (11 outputs left out)

100%|██████████| 375/375 [02:44<00:00,  2.28it/s]


Training loss after epoch 5: 1.0836081447601318


100%|██████████| 125/125 [00:13<00:00,  9.09it/s]


Validation loss after epoch 5: 1.1206074299812316
Validation loss decreased to 1.1206074299812316. Saving model...
Best model saved.


# Validation Scores from best model path and calculating Rouge and BLEU scores

In [46]:
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
import torch

# Load the model and tokenizer
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.load_state_dict(torch.load("/kaggle/input/llm3_mt5_bestmodel/pytorch/default/1/best_model (7).pth"))
model.eval()  # Set to evaluation mode
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

# Move the model to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


<ipython-input-46-b9f48f6d3554>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/input/llm3_mt5_bestmodel/pytorch/default/1/best_mo

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [47]:
# from torch.utils.data import DataLoader

# val_dataloader = torch.utils.data.DataLoader(
#     tokenized_dataset_eval["train"],
#     batch_size=8,  # Adjust batch size as needed
#     collate_fn=lambda batch: {
#         key: torch.tensor([item[key] for item in batch], dtype=torch.long) 
#         if isinstance(batch[0].get(key), list) else batch[0][key] 
#         for key in batch[0]
#     }
# )


In [108]:
predictions = []
references = []

for batch in tqdm(val_dataloader):
    batch = {k: v.to(training_args.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}  

    batch = {k: v for k, v in batch.items() if k in ["input_ids", "attention_mask", "labels"]}
 
    # Generate predictions
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask,
                             max_length=30,num_beams=4,
                             repetition_penalty=2.0,  # Penalty for repetition
                            length_penalty=1.0,  # Controls the length of the generated text 

                                                )  # Adjust max_length as needed
    
    # Decode predictions and references
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    refs = tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)
    
    predictions.extend(preds)
    references.extend(refs)


100%|██████████| 125/125 [00:56<00:00,  2.21it/s]


In [109]:
correct_preds = [i for i, (pred, ref) in enumerate(zip(predictions, references)) if pred == ref]
incorrect_preds = [i for i, (pred, ref) in enumerate(zip(predictions, references)) if pred != ref]
print(f"Number of correct predictions: {len(correct_preds)}")
print(f"Number of incorrect predictions: {len(incorrect_preds)}")


Number of correct predictions: 1
Number of incorrect predictions: 999


In [110]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# Calculate ROUGE-L scores
rouge_scores = [scorer.score(ref, pred) for ref, pred in zip(references, predictions)]
avg_rouge_l = sum([score['rougeL'].fmeasure for score in rouge_scores]) / len(rouge_scores)
print(f"Average ROUGE-L Score: {avg_rouge_l}")

# Prepare BLEU inputs
references_bleu = [[ref.split()] for ref in references]
predictions_bleu = [pred.split() for pred in predictions]

# Calculate BLEU score
bleu_score = corpus_bleu(references_bleu, predictions_bleu)
print(f"BLEU Score: {bleu_score}")


Average ROUGE-L Score: 0.1890685331550497
BLEU Score: 0.026511074713852032


# Generating Predictions for Test data

In [111]:
def preprocess_function_test(examples):
    inputs = [preprocess_text(text) for text in examples["News Article"]]

    model_inputs = tokenizer(inputs, max_length=256, truncation=True,padding="max_length",
                             return_tensors="pt")
    return model_inputs

In [112]:
test_dataset = load_dataset("csv",data_files="/kaggle/input/test-data/UNLABELLED_TEST.csv")

In [113]:
test_dataset["train"]["ID"][0]

'TEST_1_1'

In [114]:
test_split = test_dataset["train"]

tokenized_test_data = test_split.map(preprocess_function_test, batched=True)

In [115]:
# tokenized_test_data["train"]["ID"][3]

In [116]:
from torch.utils.data import DataLoader
import pandas as pd
from tqdm import tqdm

def preprocess_and_generate_predictions_with_dataloader(tokenized_test_data, model, tokenizer, output_file="predictions.csv", batch_size=8):
    # Ensure the dataset contains the "ID" field
    if "ID" not in tokenized_test_data.column_names:
        raise ValueError("The 'ID' field is missing from the test dataset. Ensure it's included before running predictions.")

    # Create DataLoader
    test_dataloader = DataLoader(
        tokenized_test_data,
        batch_size=batch_size,
        collate_fn=lambda batch: {
            key: torch.tensor([item[key] for item in batch], dtype=torch.long)
            if key in ["input_ids", "attention_mask", "labels"] else [item[key] for item in batch]
            for key in batch[0]
        }
    )

    predictions = []
    ids = []

    model.eval()

    # Generate predictions
    with torch.no_grad():
        for batch in tqdm(test_dataloader):
            # Extract IDs separately
            batch_ids = batch["ID"]
            ids.extend(batch_ids)

            # Move tensors to device
            input_ids = batch["input_ids"].to(model.device)
            attention_mask = batch["attention_mask"].to(model.device)

            # Generate predictions
            outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask,
                             max_length=25,num_beams=4,
                             repetition_penalty=2.0,  # Penalty for repetition
                            length_penalty=1.0,  # Controls the length of the generated text                             
                            )
            # outputs = model.generate(input_ids=input_ids, 
            #                          attention_mask=attention_mask, 
            #                          max_length=30)

            # Decode predictions
            decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            predictions.extend(decoded_preds)

    # Save predictions to a CSV file
    result_df = pd.DataFrame({'ID': ids, 'Prediction': predictions})
    result_df.to_csv(output_file, index=False)
    print(f"Predictions saved to {output_file}")


In [117]:
# Generate predictions for the test dataset
preprocess_and_generate_predictions_with_dataloader(tokenized_test_data, model, tokenizer,output_file="test_predictions.csv")

100%|██████████| 57/57 [00:24<00:00,  2.29it/s]

Predictions saved to test_predictions.csv
